# Healthcare Patient No-Show Analysis

## Notebook 2: Data Cleaning & Feature Engineering

### Objective

Raw healthcare data often contains inconsistencies, incorrect values, and unsuitable data types that must be corrected before analysis.

In this notebook, we will:

- Inspect data quality
- Rename inconsistent columns
- Convert date columns
- Handle invalid records
- Create new analytical features
- Prepare the dataset for Exploratory Data Analysis (EDA)

---

### Tools Used

- Python
- Pandas
- NumPy

In [1]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


## Load Dataset

Load the original dataset for cleaning.

In [2]:
# Load dataset

df = pd.read_csv("medical_appointment_no_shows.csv")

df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


## Create Working Copy

To preserve the original dataset, all cleaning operations will be performed on a copy.

In [3]:
df_clean = df.copy()

# Rename Columns

Some column names contain spelling inconsistencies.

Renaming improves readability and code quality.

In [4]:
df_clean.rename(columns={
    "Hipertension":"Hypertension",
    "Handcap":"Handicap",
    "No-show":"No_Show"
}, inplace=True)

df_clean.columns

Index(['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay',
       'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hypertension',
       'Diabetes', 'Alcoholism', 'Handicap', 'SMS_received', 'No_Show'],
      dtype='object')

# Convert Date Columns

The appointment dates are currently stored as text.

Converting them into datetime format allows date calculations.

In [5]:
df_clean["ScheduledDay"] = pd.to_datetime(df_clean["ScheduledDay"])

df_clean["AppointmentDay"] = pd.to_datetime(df_clean["AppointmentDay"])

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   PatientId       110527 non-null  float64            
 1   AppointmentID   110527 non-null  int64              
 2   Gender          110527 non-null  object             
 3   ScheduledDay    110527 non-null  datetime64[ns, UTC]
 4   AppointmentDay  110527 non-null  datetime64[ns, UTC]
 5   Age             110527 non-null  int64              
 6   Neighbourhood   110527 non-null  object             
 7   Scholarship     110527 non-null  int64              
 8   Hypertension    110527 non-null  int64              
 9   Diabetes        110527 non-null  int64              
 10  Alcoholism      110527 non-null  int64              
 11  Handicap        110527 non-null  int64              
 12  SMS_received    110527 non-null  int64              
 13  No_Show       

# Check Invalid Ages

Patient age cannot be negative.

We will identify and remove invalid records.

In [6]:
df_clean[df_clean["Age"]<0]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,No_Show
99832,4.659432e+14,5775010,F,2016-06-06 08:58:13+00:00,2016-06-06 00:00:00+00:00,-1,ROMÃO,0,0,0,0,0,0,No


In [7]:
df_clean[df_clean["Age"]<0].shape

(1, 14)

### Observation

Only a very small number of records contain invalid age values.

These records will be removed.

In [8]:
df_clean = df_clean[df_clean["Age"]>=0]

df_clean.shape

(110526, 14)

# Create Waiting Time Feature

Waiting time is the number of days between scheduling an appointment and the actual appointment.

This feature is expected to influence patient no-show behavior.

In [10]:
df_clean["Waiting_Days"] = (
    df_clean["AppointmentDay"] - df_clean["ScheduledDay"]
).dt.days

df_clean[["ScheduledDay","AppointmentDay","Waiting_Days"]].head()

,ScheduledDay,AppointmentDay,Waiting_Days
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,-1
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,-1
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,-1
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,-1
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,-1


In [11]:
df_clean["Waiting_Days"].describe()

,Waiting_Days
count,110526.000000
mean,9.183794
std,15.255034
min,-7.000000
25%,-1.000000
50%,3.000000
75%,14.000000
max,178.000000


In [12]:
df_clean[df_clean["Waiting_Days"]<0]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,No_Show,Waiting_Days
0,2.987250e+13,5642903,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,0,No,-1
1,5.589978e+14,5642503,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,0,No,-1
2,4.262962e+12,5642549,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,0,No,-1
3,8.679512e+11,5642828,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No,-1
4,8.841186e+12,5642494,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,0,No,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110511,8.235996e+11,5786742,F,2016-06-08 08:50:20+00:00,2016-06-08 00:00:00+00:00,14,MARIA ORTIZ,0,0,0,0,0,0,No,-1
110512,9.876246e+13,5786368,F,2016-06-08 08:20:01+00:00,2016-06-08 00:00:00+00:00,41,MARIA ORTIZ,0,0,0,0,0,0,No,-1
110513,8.674778e+13,5785964,M,2016-06-08 07:52:55+00:00,2016-06-08 00:00:00+00:00,2,ANTÔNIO HONÓRIO,0,0,0,0,0,0,No,-1
110514,2.695685e+12,5786567,F,2016-06-08 08:35:31+00:00,2016-06-08 00:00:00+00:00,58,MARIA ORTIZ,0,0,0,0,0,0,No,-1


# Create Age Groups

Grouping patients by age makes analysis easier and produces more meaningful visualizations.

In [14]:
bins=[0,12,18,35,60,120]

labels=[
    "Child",
    "Teen",
    "Young Adult",
    "Adult",
    "Senior"
]

df_clean["Age_Group"]=pd.cut(
    df_clean["Age"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [15]:
df_clean["Age_Group"].value_counts()

,count
Age_Group,
Adult,37761
Young Adult,24137
Child,21036
Senior,19762
Teen,7830


# Save Cleaned Dataset

The cleaned dataset will be used in subsequent notebooks.

In [16]:
df_clean.to_csv(
    "cleaned_patient_data.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


# Summary

### Data Cleaning Completed

✔ Renamed inconsistent columns

✔ Converted date columns

✔ Removed invalid age records

✔ Created Waiting_Days feature

✔ Created Age_Group feature

✔ Saved cleaned dataset

The dataset is now ready for Exploratory Data Analysis.